## Primero cargamos los datos y librerias

In [36]:
# =========================
# CARGA DE LIBRERÍAS Y DATOS
# =========================

import warnings
warnings.filterwarnings("ignore")  # Oculta warnings para no ensuciar la salida

from pathlib import Path  # Para manejar rutas de archivos de forma segura
import numpy as np  # Librería numérica (no se usa mucho aquí, pero es estándar)
import pandas as pd  # Librería principal para análisis de datos
import matplotlib.pyplot as plt  # Para gráficos (aunque aquí no se usa aún)

# Definimos la carpeta donde está el dataset
DATA_DIR = Path("../datasets")

# Hacemos que pandas muestre más columnas si existen
pd.set_option("display.max_columns", 50)

# Cargamos el dataset Iris en un DataFrame
df = pd.read_csv(DATA_DIR / "iris.csv")

# Mostramos las primeras filas para ver cómo es el dataset
display(df.head())

# Información general: tipos de datos, nulos, memoria, etc.
df.info()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.10,3.50,1.40,0.20,setosa
1,4.90,3.00,1.40,0.20,setosa
2,4.70,3.20,1.30,0.20,setosa
3,4.60,3.10,1.50,0.20,setosa
4,5.00,3.60,1.40,0.20,setosa


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


## Limpieza basica y tipos de datos

In [37]:
# =========================
# LIMPIEZA BÁSICA DE DATOS
# =========================

# Creamos una copia para no modificar el original
df_clean = df.copy()

# Contamos filas duplicadas
print("Duplicados:", df_clean.duplicated().sum())

# Eliminamos duplicados
df_clean = df_clean.drop_duplicates()

# Mostramos nulos por columna antes de limpiar
print("\nNulos por columna:")
print(df_clean.isnull().sum())

# Seleccionamos columnas numéricas
num_cols = df_clean.select_dtypes(include="number").columns

# Rellenamos nulos numéricos con la mediana (más robusta que la media)
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

# Rellenamos nulos de la variable categórica con "desconocido"
df_clean["species"] = df_clean["species"].fillna("desconocido")

# Comprobación final de nulos después de la limpieza
print("\nNulos después:")
print(df_clean.isnull().sum())


Duplicados: 3

Nulos por columna:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

Nulos después:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


## Resumen estadistico limpio

In [39]:
# Generamos estadísticas generales (numéricas y categóricas)
desc = df_clean.describe(include="all")

# Redondeamos valores numéricos para que sea más legible
desc = desc.apply(lambda col: col.round(2) if col.dtype != "object" else col)

# Mostramos el resumen
display(desc)


,sepal_length,sepal_width,petal_length,petal_width,species
count,147.00,147.00,147.00,147.00,147
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,versicolor
freq,NaN,NaN,NaN,NaN,50
mean,5.86,3.06,3.78,1.21,NaN
std,0.83,0.44,1.76,0.76,NaN
min,4.30,2.00,1.00,0.10,NaN
25%,5.10,2.80,1.60,0.30,NaN
50%,5.80,3.00,4.40,1.30,NaN
75%,6.40,3.30,5.10,1.80,NaN


## Distribucion de especies

In [40]:
# =========================
# DISTRIBUCIÓN DE ESPECIES
# =========================

# Conteo absoluto de cada especie
print("\nDistribución de especies:")
print(df_clean["species"].value_counts())

# Conteo relativo (proporciones)
print("\nDistribución relativa:")
print(df_clean["species"].value_counts(normalize=True).round(2))


Distribución de especies:
species
versicolor    50
virginica     49
setosa        48
Name: count, dtype: int64

Distribución relativa:
species
versicolor   0.34
virginica    0.33
setosa       0.33
Name: proportion, dtype: float64


## Estadisticas numericas y categoricas

In [41]:
# =========================
# ESTADÍSTICAS POR TIPO DE VARIABLE
# =========================

# Estadísticas solo de variables numéricas
num_desc = df_clean.describe().round(2)

# Estadísticas solo de variables categóricas
cat_desc = df_clean.describe(include="object")

print("\nVARIABLES NUMÉRICAS:")
display(num_desc)

print("\nVARIABLES CATEGÓRICAS:")
display(cat_desc)


VARIABLES NUMÉRICAS:


,sepal_length,sepal_width,petal_length,petal_width
count,147.00,147.00,147.00,147.00
mean,5.86,3.06,3.78,1.21
std,0.83,0.44,1.76,0.76
min,4.30,2.00,1.00,0.10
25%,5.10,2.80,1.60,0.30
50%,5.80,3.00,4.40,1.30
75%,6.40,3.30,5.10,1.80
max,7.90,4.40,6.90,2.50



VARIABLES CATEGÓRICAS:


,species
count,147
unique,3
top,versicolor
freq,50


## Estadisticas por especie

In [ ]:
# =========================
# ESTADÍSTICAS POR ESPECIE
# =========================

# Media de variables numéricas por especie
print("\nMedia por especie:")
display(df_clean.groupby("species").mean(numeric_only=True).round(2))


Media por especie:


,sepal_length,sepal_width,petal_length,petal_width
species,,,,
setosa,5.01,3.43,1.46,0.25
versicolor,5.94,2.77,4.26,1.33
virginica,6.60,2.98,5.56,2.03


## Estadisticas por grupos

In [43]:
# =========================
# ESTADÍSTICAS COMPLETAS POR GRUPO
# =========================

# Para cada especie calculamos varias métricas:
# media, desviación estándar, mínimo y máximo
print("\nEstadísticas completas por especie:")
display(df_clean.groupby("species").agg(["mean", "std", "min", "max"]).round(2))


Estadísticas completas por especie:


sepal_length                sepal_width                 \
                   mean  std  min  max        mean  std  min  max   
species                                                             
setosa             5.01 0.36 4.30 5.80        3.43 0.38 2.30 4.40   
versicolor         5.94 0.52 4.90 7.00        2.77 0.31 2.00 3.40   
virginica          6.60 0.63 4.90 7.90        2.98 0.32 2.20 3.80   

           petal_length                petal_width                 
                   mean  std  min  max        mean  std  min  max  
species                                                            
setosa             1.46 0.18 1.00 1.90        0.25 0.11 0.10 0.60  
versicolor         4.26 0.47 3.00 5.10        1.33 0.20 1.00 1.80  
virginica          5.56 0.55 4.50 6.90        2.03 0.28 1.40 2.50